# Run the UK Legal SDK Evaluation

This is the main participant evaluation notebook. Run it after each workshop stage to measure answer quality and retain the standard Fabric Data Agent SDK results plus SQL, DAX, KQL, and source evidence when the companion run-step data provides it.

## Required Fabric setup

Before running the notebook:

1. Import both `NB_Deploy_Data_Agent_Hackathon.ipynb` and this notebook into the Fabric workspace.
2. Run the deployment notebook to create or reset the workshop environment.
3. In this notebook's **Explorer** pane, select **Add data items**, attach `LegalFirmDemo`, and make it the **default Lakehouse**.
4. Set `WORKSPACE_NAME` and `DATA_AGENT_STAGE` for the deployed agent.
5. Change only `SNAPSHOT_NAME` as you move through the workshop.
6. Run all cells after each stage.

The SDK writes each snapshot to its own Delta result table and companion `_steps` table; you do not create it manually. The final section reads all available snapshot tables and builds a question-by-step comparison. Missing future snapshots are skipped until you run them.

`get_evaluation_details()` supplies the official question, expected answer, actual answer, judgement, and thread details. Generated queries and selected-source traces depend on companion run-step data. The notebook warns clearly when that evidence is unavailable; it never claims that missing SQL or DAX was captured.

In [ ]:
# 1. Install the pinned SDK dependencies, then restart the Fabric Python interpreter.

import subprocess
import sys

packages = [
    "fabric-data-agent-sdk==0.1.30a0",
    "pandas",
    "typing_extensions>=4.12.2",
    "PyJWT>=2.6.0",
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-U", *packages]
)

verification = subprocess.run(
    [
        sys.executable,
        "-c",
        "from fabric.dataagent.evaluation import "
        "evaluate_data_agent, get_evaluation_details, get_evaluation_summary; "
        "from fabric.dataagent.evaluation._storage import _get_data; "
        "from typing_extensions import Sentinel; "
        "print('Fabric Data Agent evaluation SDK dependencies verified')",
    ],
    capture_output=True,
    text=True,
)
if verification.returncode != 0:
    raise RuntimeError(
        "The packages installed, but a fresh Python process could not import them.\n"
        f"{verification.stderr.strip()}"
    )
print(verification.stdout.strip())
print("Restarting the Fabric Python interpreter; execution will continue in the next cell.")
notebookutils.session.restartPython()  # type: ignore[name-defined]

## 2. Configure and verify the restarted kernel

In [ ]:
import importlib

DOMAIN_PROFILE = "uk-legal"
DEFAULT_LAKEHOUSE_NAME = "LegalFirmDemo"
AGENT_NAME = "LegalFirmAgent"
DOMAIN_TOKEN = DOMAIN_PROFILE.replace("-", "_")

fabric_evaluation = importlib.import_module("fabric.dataagent.evaluation")
fabric_runtime = importlib.import_module("fabric.dataagent._fabric_runtime")
evaluation_storage = importlib.import_module("fabric.dataagent.evaluation._storage")

print("Current notebook kernel import verified:", fabric_evaluation.__name__)
fabric_context = fabric_runtime.get_fabric_context()
default_lakehouse_id = fabric_context.get("trident.lakehouse.id")
default_lakehouse_filesystem = fabric_context.get("fs.defaultFS")
if not default_lakehouse_id or not default_lakehouse_filesystem:
    raise RuntimeError(
        "No default Lakehouse is attached. In the Explorer pane, select Add data items, "
        f"attach {DEFAULT_LAKEHOUSE_NAME}, set it as the default Lakehouse, and rerun from this cell."
    )
print("Domain profile:", DOMAIN_PROFILE)
print("Default Lakehouse verified:", default_lakehouse_id)

# Change only this value as the workshop progresses.
SNAPSHOT_NAME = "step1_baseline"
SNAPSHOT_PLAN = {
    "step1_baseline": "challenge",
    "step2_prep_ai": "challenge",
    "step3_lakehouse_added": "challenge",
    "step4_lakehouse_tuned": "challenge",
    "step5_final": "challenge",
    "step5_routing": "routing",
}
if SNAPSHOT_NAME not in SNAPSHOT_PLAN:
    raise ValueError(
        f"SNAPSHOT_NAME must be one of {list(SNAPSHOT_PLAN)}; received {SNAPSHOT_NAME!r}."
    )
DATASET_NAME = SNAPSHOT_PLAN[SNAPSHOT_NAME]
INCLUDE_PARAPHRASES = DATASET_NAME == "challenge"

WORKSPACE_NAME = "Hackathon"
DATA_AGENT_STAGE = "sandbox"  # "sandbox"/"draft" before publish; "production" after publish
CRITIC_PROMPT = ""
TABLE_NAME = f"{DOMAIN_TOKEN}_evaluation_{SNAPSHOT_NAME}"
OUTPUT_PATH = f"{DOMAIN_PROFILE}_{SNAPSHOT_NAME}_sdk_evaluation_results.json"
SAVE_EVIDENCE_CSV = True
SAVE_COMPARISON_CSV = True

REPOSITORY_OWNER = "Limaoncloud"
REPOSITORY_NAME = "fabric-data-agent-hackathon"
REPOSITORY_REF = "dev"

print("Snapshot:", SNAPSHOT_NAME)
print("Dataset:", DATASET_NAME)
print("Result table:", f"eval_result.{TABLE_NAME}")
print("Run-step table:", f"eval_result.{TABLE_NAME}_steps")

## 3. Download the question dataset

Download the versioned challenge or routing dataset directly from this repository. No repository Python module is required.

In [ ]:
import json
import tempfile
from pathlib import Path

import pandas as pd
import requests

raw_base_url = (
    f"https://raw.githubusercontent.com/{REPOSITORY_OWNER}/"
    f"{REPOSITORY_NAME}/{REPOSITORY_REF}"
)
dataset_url = f"{raw_base_url}/evaluation/{DATASET_NAME}/{DOMAIN_PROFILE}.json"

dataset_response = requests.get(dataset_url, timeout=180)
dataset_response.raise_for_status()
dataset = dataset_response.json()

prepared_queries = []
for item in dataset["evaluation_queries"]:
    expected_answer = item.get("sdk_expected_answer", item["ground_truth_answer"])
    original = {
        **item,
        "original_id": item["id"],
        "variant": "original",
        "sdk_expected_answer": expected_answer,
    }
    prepared_queries.append(original)
    if INCLUDE_PARAPHRASES:
        prepared_queries.append(
            {
                **item,
                "id": f"{item['id']}-P",
                "original_id": item["id"],
                "question": item["paraphrase"],
                "variant": "paraphrase",
                "sdk_expected_answer": expected_answer,
            }
        )

dataset["evaluation_queries"] = prepared_queries
dataset["metadata"]["total_queries"] = len(prepared_queries)

prompt_metadata_df = pd.DataFrame(
    [
        {
            "question_id": item["id"],
            "original_id": item["original_id"],
            "variant": item["variant"],
            "question": item["question"],
            "expected_source": item.get("expected_source", ""),
            "expected_measure_or_object": item.get("expected_measure", item.get("expected_object", "")),
            "sdk_expected_answer": item["sdk_expected_answer"],
        }
        for item in prepared_queries
    ]
)

dataset_path = Path(tempfile.gettempdir()) / (
    f"{SNAPSHOT_NAME}_{DATASET_NAME}_{DOMAIN_PROFILE}.json"
)
dataset_path.write_text(json.dumps(dataset, indent=2), encoding="utf-8")

print("Dataset URL:", dataset_url)
print("SDK prompts:", len(prepared_queries))
print("Dataset saved to:", dataset_path)
display(prompt_metadata_df)

## 4. Run the SDK-backed snapshot

Run this notebook after the selected workshop stage. Challenge snapshots evaluate eight questions plus eight paraphrases, for 16 prompts. The separate routing snapshot evaluates its three routing questions without paraphrases.

In [ ]:
valid_data_agent_stages = {"sandbox", "draft", "production"}
data_agent_stage = DATA_AGENT_STAGE.strip().lower()
if data_agent_stage not in valid_data_agent_stages:
    raise ValueError(
        f"DATA_AGENT_STAGE must be one of {sorted(valid_data_agent_stages)}; "
        f"received {DATA_AGENT_STAGE!r}."
    )

sdk_input_df = pd.DataFrame(
    {
        "question": [item["question"] for item in prepared_queries],
        "expected_answer": [item["sdk_expected_answer"] for item in prepared_queries],
    }
)

evaluation_kwargs = {
    "workspace_name": WORKSPACE_NAME or None,
    "table_name": TABLE_NAME,
    "data_agent_stage": data_agent_stage,
}
if CRITIC_PROMPT:
    evaluation_kwargs["critic_prompt"] = CRITIC_PROMPT

evaluation_id = fabric_evaluation.evaluate_data_agent(
    sdk_input_df,
    AGENT_NAME,
    **evaluation_kwargs,
)
if evaluation_id is None:
    raise RuntimeError(
        "The Fabric SDK did not return an evaluation ID. Check the agent name, workspace, "
        "stage, and default Lakehouse, then rerun this cell."
    )

sdk_summary_df = fabric_evaluation.get_evaluation_summary(
    table_name=TABLE_NAME,
    verbose=False,
)
sdk_details_df = fabric_evaluation.get_evaluation_details(
    evaluation_id=evaluation_id,
    table_name=TABLE_NAME,
    get_all_rows=True,
    verbose=False,
)
if sdk_details_df is None or sdk_details_df.empty:
    raise RuntimeError(
        f"The evaluation ran, but no detail rows were returned. Confirm that {DEFAULT_LAKEHOUSE_NAME} "
        "is attached as the default Lakehouse and rerun from section 2."
    )

print("Evaluation ID:", evaluation_id)
display(sdk_summary_df)
display(sdk_details_df)

## 5. Capture answers and generated query evidence

The standard SDK detail frame is preserved in full. The companion `${TABLE_NAME}_steps` data is read separately and joined by the SDK row `id` to expose generated SQL, DAX, or KQL and selected-source traces where available.

Run-step schemas can vary by SDK release and tool path. Missing fields remain `Not captured`, and the cell prints a warning rather than treating expected source metadata as observed evidence.

In [ ]:
import ast

STEP_COLUMNS = [
    "id",
    "function_names",
    "function_queries",
    "function_outputs",
    "sql_steps",
    "dax_steps",
    "kql_steps",
]
SOURCE_COLUMNS = [
    "selected_source",
    "source_name",
    "data_source",
    "artifact_name",
    "source",
]


def step_values(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    parsed = value
    if isinstance(value, str):
        text = value.strip()
        if not text or text.casefold() == "none":
            return []
        try:
            parsed = ast.literal_eval(text)
        except (SyntaxError, ValueError):
            parsed = text
    if not isinstance(parsed, (list, tuple, set)):
        parsed = [parsed]
    cleaned = []
    for item in parsed:
        text = str(item).strip()
        if text and text.casefold() not in {"none", "nan", "[]"} and text not in cleaned:
            cleaned.append(text)
    return cleaned


def first_observed_value(row, columns):
    for column in columns:
        if column not in row.index:
            continue
        values = step_values(row[column])
        if values:
            return " | ".join(values)
    return ""


def build_evidence(details_df, steps_df, metadata_df, snapshot_name, dataset_name):
    details = details_df.copy()
    available_step_columns = [column for column in STEP_COLUMNS if column in steps_df.columns]
    steps = steps_df[available_step_columns].copy() if available_step_columns else pd.DataFrame({"id": []})
    for column in STEP_COLUMNS:
        if column not in steps.columns:
            steps[column] = ""
    evidence = details.merge(steps[STEP_COLUMNS], on="id", how="left", suffixes=("", "_step"))
    evidence = evidence.merge(metadata_df, on="question", how="left")
    evidence.insert(0, "dataset_name", dataset_name)
    evidence.insert(0, "snapshot_name", snapshot_name)

    for column in ("sql_steps", "dax_steps", "kql_steps"):
        evidence[column] = evidence[column].apply(lambda value: "\n\n".join(step_values(value)))

    def query_type(row):
        types = [
            label
            for label, column in (("SQL", "sql_steps"), ("DAX", "dax_steps"), ("KQL", "kql_steps"))
            if row[column]
        ]
        return " + ".join(types) if types else "Not captured"

    def generated_query(row):
        sections = []
        for label, column in (("SQL", "sql_steps"), ("DAX", "dax_steps"), ("KQL", "kql_steps")):
            if row[column]:
                sections.append(f"{label}:\n{row[column]}")
        return "\n\n".join(sections) if sections else "Not captured"

    def selected_source(row):
        observed = first_observed_value(row, SOURCE_COLUMNS)
        if observed:
            return observed
        trace = " | ".join(
            first_observed_value(row, [column])
            for column in ("function_names", "function_queries", "function_outputs")
            if first_observed_value(row, [column])
        )
        known_sources = [name for name in ("LegalFirmSemanticModel", "LegalFirmDemo") if name in trace]
        return " | ".join(known_sources) if known_sources else "Not captured"

    evidence["query_type"] = evidence.apply(query_type, axis=1)
    evidence["generated_query"] = evidence.apply(generated_query, axis=1)
    evidence["selected_source"] = evidence.apply(selected_source, axis=1)
    evidence["sdk_judgement"] = (
        evidence["evaluation_judgement"] if "evaluation_judgement" in evidence else "Not captured"
    )
    evidence["thread_link"] = evidence["thread_url"] if "thread_url" in evidence else "Not captured"
    return evidence


try:
    sdk_steps_df = evaluation_storage._get_data(f"{TABLE_NAME}_steps")
except Exception as error:
    print(f"WARNING: Companion run-step data could not be read: {error}")
    sdk_steps_df = None
if sdk_steps_df is None:
    sdk_steps_df = pd.DataFrame(columns=STEP_COLUMNS)
current_ids = set(sdk_details_df["id"].astype(str))
if "id" in sdk_steps_df.columns:
    sdk_steps_df = sdk_steps_df[sdk_steps_df["id"].astype(str).isin(current_ids)]

evidence_df = build_evidence(
    sdk_details_df,
    sdk_steps_df,
    prompt_metadata_df,
    SNAPSHOT_NAME,
    DATASET_NAME,
)

EVIDENCE_COLUMNS = [
    "snapshot_name",
    "dataset_name",
    "question_id",
    "variant",
    "question",
    "expected_answer",
    "actual_answer",
    "sdk_judgement",
    "thread_link",
    "expected_source",
    "selected_source",
    "query_type",
    "generated_query",
]
missing_query_count = int((evidence_df["generated_query"] == "Not captured").sum())
missing_source_count = int((evidence_df["selected_source"] == "Not captured").sum())
if missing_query_count:
    print(
        f"WARNING: SQL/DAX/KQL was unavailable for {missing_query_count} row(s). "
        "The official SDK detail results are preserved; inspect the Data Agent thread when query evidence is required."
    )
if missing_source_count:
    print(
        f"WARNING: Selected source was unavailable for {missing_source_count} row(s). "
        "Expected source is shown separately and is not treated as observed evidence."
    )
display(evidence_df[EVIDENCE_COLUMNS])

output_path = Path(OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)
snapshot_payload = {
    "snapshot_name": SNAPSHOT_NAME,
    "dataset_name": DATASET_NAME,
    "agent_name": AGENT_NAME,
    "workspace_name": WORKSPACE_NAME or None,
    "data_agent_stage": data_agent_stage,
    "evaluation_id": str(evaluation_id),
    "table_name": TABLE_NAME,
    "steps_table_name": f"{TABLE_NAME}_steps",
    "question_count": len(prepared_queries),
    "official_summary": json.loads(sdk_summary_df.to_json(orient="records")),
    "official_details": json.loads(sdk_details_df.to_json(orient="records")),
    "official_steps": json.loads(sdk_steps_df.to_json(orient="records")),
    "evidence": json.loads(evidence_df.to_json(orient="records")),
}
output_path.write_text(json.dumps(snapshot_payload, indent=2), encoding="utf-8")
print("Snapshot JSON:", output_path.resolve())

if SAVE_EVIDENCE_CSV:
    evidence_csv_path = output_path.with_name(f"{output_path.stem}_evidence.csv")
    evidence_df.to_csv(evidence_csv_path, index=False)
    print("Evidence CSV (official columns preserved):", evidence_csv_path.resolve())

## 6. Compare all completed workshop snapshots

This section reads the latest run from every available snapshot table. It displays an overall quality trend, a question-by-step SDK judgement matrix, and a long evidence table with generated queries and selected sources where the SDK run-step evidence exposed them.

In [ ]:
def read_optional_table(table_name):
    try:
        data = evaluation_storage._get_data(table_name)
        return data if data is not None and not data.empty else None
    except Exception:
        return None


def latest_evaluation_rows(all_details):
    if all_details is None or all_details.empty:
        return None
    details = all_details.copy()
    if "run_timestamp" in details.columns:
        details["_parsed_timestamp"] = pd.to_datetime(details["run_timestamp"], errors="coerce")
        latest_row = details.sort_values("_parsed_timestamp").iloc[-1]
        details = details.drop(columns=["_parsed_timestamp"])
    else:
        latest_row = details.iloc[-1]
    if "evaluation_id" not in details.columns:
        return details
    return details[details["evaluation_id"] == latest_row["evaluation_id"]]


def load_prompt_metadata(dataset_name):
    url = f"{raw_base_url}/evaluation/{dataset_name}/{DOMAIN_PROFILE}.json"
    response = requests.get(url, timeout=180)
    response.raise_for_status()
    rows = []
    for item in response.json()["evaluation_queries"]:
        expected_answer = item.get("sdk_expected_answer", item["ground_truth_answer"])
        rows.append(
            {
                "question_id": item["id"],
                "original_id": item["id"],
                "variant": "original",
                "question": item["question"],
                "expected_source": item.get("expected_source", ""),
                "expected_measure_or_object": item.get("expected_measure", item.get("expected_object", "")),
                "sdk_expected_answer": expected_answer,
            }
        )
        if dataset_name == "challenge":
            rows.append(
                {
                    "question_id": f"{item['id']}-P",
                    "original_id": item["id"],
                    "variant": "paraphrase",
                    "question": item["paraphrase"],
                    "expected_source": item.get("expected_source", ""),
                    "expected_measure_or_object": item.get("expected_measure", item.get("expected_object", "")),
                    "sdk_expected_answer": expected_answer,
                }
            )
    return pd.DataFrame(rows)


metadata_by_dataset = {
    dataset_name: load_prompt_metadata(dataset_name)
    for dataset_name in set(SNAPSHOT_PLAN.values())
}
comparison_frames = []
summary_rows = []

for snapshot_name, dataset_name in SNAPSHOT_PLAN.items():
    snapshot_table = f"{DOMAIN_TOKEN}_evaluation_{snapshot_name}"
    latest_details = latest_evaluation_rows(read_optional_table(snapshot_table))
    if latest_details is None:
        continue
    snapshot_steps = read_optional_table(f"{snapshot_table}_steps")
    if snapshot_steps is None:
        snapshot_steps = pd.DataFrame(columns=STEP_COLUMNS)
    latest_ids = set(latest_details["id"].astype(str))
    if "id" in snapshot_steps.columns:
        snapshot_steps = snapshot_steps[snapshot_steps["id"].astype(str).isin(latest_ids)]
    snapshot_evidence = build_evidence(
        latest_details,
        snapshot_steps,
        metadata_by_dataset[dataset_name],
        snapshot_name,
        dataset_name,
    )
    comparison_frames.append(snapshot_evidence)

    judgements = snapshot_evidence["sdk_judgement"]
    normalized = judgements.astype(str).str.strip().str.casefold()
    passed = int(normalized.isin({"true", "pass", "passed", "1"}).sum())
    failed = int(normalized.isin({"false", "fail", "failed", "0"}).sum())
    total = len(snapshot_evidence)
    summary_rows.append(
        {
            "snapshot_name": snapshot_name,
            "dataset_name": dataset_name,
            "evaluation_id": str(snapshot_evidence["evaluation_id"].iloc[0]),
            "passed": passed,
            "failed": failed,
            "unclear": total - passed - failed,
            "total": total,
            "pass_rate_percent": round(100.0 * passed / total, 1) if total else 0.0,
        }
    )

if not comparison_frames:
    print("No completed snapshot tables were found.")
else:
    snapshot_summary_df = pd.DataFrame(summary_rows)
    comparison_evidence_df = pd.concat(comparison_frames, ignore_index=True)
    comparison_evidence_df["judgement"] = comparison_evidence_df["sdk_judgement"].apply(
        lambda value: "Pass"
        if str(value).strip().casefold() in {"true", "pass", "passed", "1"}
        else "Fail"
        if str(value).strip().casefold() in {"false", "fail", "failed", "0"}
        else "Unclear"
    )
    judgement_matrix_df = comparison_evidence_df.pivot_table(
        index=["original_id", "variant", "question"],
        columns="snapshot_name",
        values="judgement",
        aggfunc="first",
    ).reset_index()

    ordered_snapshots = [name for name in SNAPSHOT_PLAN if name in judgement_matrix_df.columns]
    judgement_matrix_df = judgement_matrix_df[
        ["original_id", "variant", "question", *ordered_snapshots]
    ]

    print("Snapshot quality trend")
    display(snapshot_summary_df)
    print("Question-by-step SDK judgement")
    display(judgement_matrix_df)
    print("Query and source evidence by step")
    display(comparison_evidence_df[EVIDENCE_COLUMNS])

    if SAVE_COMPARISON_CSV:
        comparison_path = Path(f"{DOMAIN_PROFILE}_all_steps_evidence.csv")
        matrix_path = Path(f"{DOMAIN_PROFILE}_all_steps_judgement_matrix.csv")
        comparison_evidence_df.to_csv(comparison_path, index=False)
        judgement_matrix_df.to_csv(matrix_path, index=False)
        print("All-step evidence CSV:", comparison_path.resolve())
        print("Judgement matrix CSV:", matrix_path.resolve())